# The Normal (Gaussian) Distribution

This notebook introduces the most famous distribution of all: the **normal distribution**, named after Carl Friedrich Gauss. It appears everywhere — statistics, science, nature, and machine learning. We build its formula step by step, meet the standard normal and standardization, and see why its CDF needs software. It is based on the tenth video of Lesson 2 (Probability Distributions).


## 1. Introduction

Plot the binomial PMF for 2 coin tosses — a few bars. Now grow $n$: by $n = 100$ the bars trace a smooth **bell curve**. That limiting curve is the normal distribution: for large $n$, the binomial is approximated by a Gaussian remarkably well.

After completing this notebook you will be able to:

- Build the normal PDF from $e^{-x^2/2}$ by shifting, widening, and normalizing.
- Use the parameters $\mu$ (center) and $\sigma$ (spread), and the notation $N(\mu, \sigma^2)$.
- Standardize any normal variable with $z = \frac{x - \mu}{\sigma}$.
- Compute normal probabilities numerically (the CDF has no closed form).


## 2. Theory

### Building the formula, one fix at a time

Start with data shaped like a bell, centered at 2 with spread 3, and try to fit the curve $e^{-x^2/2}$:

1. **Wrong center.** The curve is symmetric around 0, the data around $\mu = 2$. Replace $x$ with $x - 2$: the curve slides right.
2. **Too skinny.** The curve has spread (standard deviation) 1; the data has 3. Divide inside the exponent by $\sigma = 3$: the curve widens.
3. **Wrong height.** The area under a density must be 1; this curve's area is $\sigma\sqrt{2\pi}$. Divide by it.

The result is the normal PDF:

$$
f(x) = \frac{1}{\sigma\sqrt{2\pi}} \, e^{-\frac{(x-\mu)^2}{2\sigma^2}}
$$

### Properties and notation

- $\mu$ = the **center** (the mean); $\sigma$ = the **spread** (the standard deviation) — both studied in detail next week.
- Symmetric around $\mu$; defined and **positive on all real numbers** (just astronomically small far out).
- Notation: $X \sim N(\mu, \sigma^2)$ — by convention the second parameter is $\sigma^2$ (the *variance*). Since $\sigma > 0$, the two carry the same information.

### The standard normal and standardization

$\mu = 0, \sigma = 1$ gives the **standard normal** — the simplified density $\frac{1}{\sqrt{2\pi}}e^{-x^2/2}$ makes calculations easier. Any normal variable converts to it:

$$
Z = \frac{X - \mu}{\sigma} \sim N(0, 1)
$$

(subtract the center, divide by the spread — the reverse of the two fixes above). Standardization lets you **compare variables of different magnitudes** on one scale.

### The CDF

Starts at 0 (at $-\infty$), ends at 1 (at $+\infty$) — but the area has **no closed-form formula**. Historically people used printed tables; today software computes it numerically.


## 3. Worked Example: standardizing

$X \sim N(2, 2.5^2)$. To standardize:

1. subtract the mean: $X - 2$ (recenter at 0);
2. divide by the standard deviation: $Z = \frac{X - 2}{2.5} \sim N(0, 1)$.

So e.g. the value $x = 7$ becomes $z = \frac{7-2}{2.5} = 2$ — "two standard deviations above the mean," a statement comparable across any normal variable.


## 4. Python Examples


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb, erf, sqrt

rng = np.random.default_rng(seed=42)

def normal_pdf(x, mu=0.0, sigma=1.0):
    return np.exp(-((x - mu) ** 2) / (2 * sigma ** 2)) / (sigma * np.sqrt(2 * np.pi))

def normal_cdf(x, mu=0.0, sigma=1.0):
    return 0.5 * (1 + erf((x - mu) / (sigma * sqrt(2))))

# numpy compatibility: trapezoid was renamed from trapz in numpy 2.0
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz


### The binomial becomes a bell

The video's opening: Binomial(n, 0.5) PMFs for growing n, with the fitted normal on top.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, n in zip(axes, [2, 10, 100]):
    ks = np.arange(n + 1)
    pmf = np.array([comb(n, int(k)) * 0.5 ** n for k in ks])
    ax.bar(ks, pmf, color="#4e79a7", edgecolor="white",
           width=0.9, label=f"Binomial({n}, 0.5)")
    mu, sigma = n / 2, np.sqrt(n) / 2
    xs = np.linspace(mu - 4 * sigma, mu + 4 * sigma, 300)
    ax.plot(xs, normal_pdf(xs, mu, sigma), color="#f28e2b", linewidth=2,
            label="normal fit")
    ax.set_title(f"n = {n}")
    ax.set_xlabel("heads")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis="y")

plt.suptitle("As n grows, the binomial traces the Gaussian bell")
plt.tight_layout()
plt.show()


### The video's three fixes, step by step


In [ ]:
mu, sigma = 2, 3
data = rng.normal(mu, sigma, size=200_000)
xs = np.linspace(-10, 14, 600)

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=False)
steps = [
    (np.exp(-xs**2 / 2), "1. raw e^(-x²/2):\nwrong center, too skinny, too tall"),
    (np.exp(-(xs - mu)**2 / 2), "2. shift x → x-μ:\ncentered, still skinny & tall"),
    (np.exp(-(xs - mu)**2 / (2 * sigma**2)) / (sigma * np.sqrt(2 * np.pi)),
     "3. widen by σ and divide by σ√(2π):\na perfect fit"),
]

for ax, (curve, title) in zip(axes, steps):
    ax.hist(data, bins=80, density=True, color="#9ecae1", edgecolor="white")
    ax.plot(xs, curve, color="#f28e2b", linewidth=2)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("x")
    ax.grid(True, alpha=0.3)

plt.suptitle("Fitting the bell: shift the center, widen the curve, fix the area")
plt.tight_layout()
plt.show()


### Standardization in action


In [ ]:
x = rng.normal(2, 2.5, size=100_000)   # X ~ N(2, 2.5²)
z = (x - 2) / 2.5                       # standardize

print(f"X: center ≈ {x.mean():.3f}, spread ≈ {x.std():.3f}")
print(f"Z: center ≈ {z.mean():.3f}, spread ≈ {z.std():.3f}  -> N(0, 1)")
print(f"x = 7 standardizes to z = {(7 - 2) / 2.5}")


### The CDF, numerically

No closed form exists — `erf` under the hood is a numerical approximation, the modern replacement for the old printed tables.


In [ ]:
for val in [-1, 0, 1, 2]:
    print(f"Standard normal: P(Z <= {val:2d}) = {normal_cdf(val):.4f}")

# the famous 68-95-99.7 rule
for k in [1, 2, 3]:
    p = normal_cdf(k) - normal_cdf(-k)
    print(f"P(within {k}σ of μ) = {p:.4f}")


## 5. Visualizations

### Different μ and σ


In [ ]:
xs = np.linspace(-10, 12, 600)

plt.figure(figsize=(9, 5))
for mu, sigma in [(0, 1), (0, 2), (3, 1), (3, 0.5)]:
    plt.plot(xs, normal_pdf(xs, mu, sigma), linewidth=2,
             label=f"N({mu}, {sigma}²)")
plt.title("μ slides the bell; σ widens or sharpens it")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


### PDF and CDF of the standard normal


In [ ]:
xs = np.linspace(-4, 4, 400)
cdf_vals = np.array([normal_cdf(v) for v in xs])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(xs, normal_pdf(xs), color="#4e79a7", linewidth=2)
mask = xs <= 1
axes[0].fill_between(xs[mask], normal_pdf(xs[mask]), color="#4e79a7", alpha=0.4)
axes[0].set_title(f"PDF: area up to 1 = {normal_cdf(1):.3f}")
axes[0].set_xlabel("z")
axes[0].set_ylabel("f(z)")

axes[1].plot(xs, cdf_vals, color="#e15759", linewidth=2)
axes[1].plot(1, normal_cdf(1), "ko")
axes[1].annotate(f"F(1) = {normal_cdf(1):.3f}", (1, normal_cdf(1)),
                 textcoords="offset points", xytext=(10, -12))
axes[1].set_title("CDF: 0 at -∞, 1 at +∞, no closed form")
axes[1].set_xlabel("z")
axes[1].set_ylabel("F(z)")

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### The whole story on one page

Everything above in a single sheet: the binomial turning into a bell, the three fixes that build the formula, the effect of μ and σ, standardization, and CDF values.

<img src="img/10_normal_distribution.png" alt="Concept sheet for this notebook" width="900">

## 6. Exercises

### Basic

1. Adult height is roughly $N(170, 10^2)$ (cm). Standardize a height of 190 cm. How many standard deviations above the mean is it?
2. In $N(\mu, \sigma^2)$ notation, what distribution is $N(5, 9)$ — what are its mean and standard deviation?

### Intermediate

3. Using `normal_cdf`, find the probability that a height from $N(170, 10^2)$ is between 160 and 180 cm. Which famous rule does this match?
4. IQ scores follow $N(100, 15^2)$. What fraction of people score above 130?

### Advanced

5. Verify the video's claim numerically: the area under $e^{-(x-\mu)^2/(2\sigma^2)}$ equals $\sigma\sqrt{2\pi}$ (try $\sigma = 3$), which is why the normalizing constant is what it is.


## 7. Solutions


In [ ]:
# Exercise 1
print(f"1.  z = (190-170)/10 = {(190-170)/10} — two standard deviations above.")

# Exercise 2
print(f"2.  N(5, 9): mean 5, variance 9 -> standard deviation 3.")

# Exercise 3
p = normal_cdf(180, 170, 10) - normal_cdf(160, 170, 10)
print(f"3.  P(160 <= X <= 180) = {p:.4f} — the 68% (1σ) rule.")

# Exercise 4
print(f"4.  P(IQ > 130) = 1 - F(130) = {1 - normal_cdf(130, 100, 15):.4f} "
      "(about 2.3%)")

# Exercise 5
sigma, mu = 3, 0
xs = np.linspace(-40, 40, 200_001)
area = np.trapezoid(np.exp(-(xs - mu) ** 2 / (2 * sigma ** 2)), xs)
print(f"5.  Numerical area = {area:.5f}; σ√(2π) = {sigma * np.sqrt(2*np.pi):.5f} ✓")


## 8. Conclusion

- The normal PDF $f(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-(x-\mu)^2/(2\sigma^2)}$ is $e^{-x^2/2}$ after three fixes: shift to $\mu$, widen by $\sigma$, normalize the area.
- $X \sim N(\mu, \sigma^2)$: symmetric bell centered at $\mu$, positive on all of $\mathbb{R}$; the second parameter is the variance $\sigma^2$.
- **Standardization** $z = \frac{x-\mu}{\sigma}$ maps any normal to $N(0,1)$ and makes different scales comparable.
- The CDF has no formula — software (or old-fashioned tables) computes the areas; the 68-95-99.7 rule covers the common cases.
- Heights, weights, IQs, channel noise, sums of many independent effects, and countless ML model assumptions — all normal. Why sums of anything become normal is the central limit theorem, later in the course.
